# KV Cache

**Tags:** optimization, memory
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/kv-cache.md

## TL;DR

Cache Key-Value matrices from previous tokens to avoid redundant computation during generation. Trade: store cache (memory) for faster decoding (no recomputation). Critical for inference speed; enables longer sequences without quadratic cost.

## Core Intuition

Autoregressive generation means predicting token 1, then token 2 (using token 1), then token 3 (using tokens 1+2), etc. Without KV cache, computing attention for token T requires recomputing attention over ALL previous tokens (T-1 times). Store the K and V matrices once, reuse them → O(1) per token instead of O(T).

## How It Works

**Without KV Cache (Naive):**
```
Token 1: compute Q1, K1, V1; output logits
Token 2: recompute Q1, K1, V1 (from scratch!), then compute Q2, K2, V2
Token 3: recompute Q1, K1, V1, Q2, K2, V2 (from scratch!), then compute Q3, K3, V3
...
Total: O(T^2) computation (quadratic)
```

**With KV Cache:**
```
Token 1: compute Q1, K1, V1; save K1, V1 in cache; output logits
Token 2: retrieve cached K1, V1; compute only Q2, K2, V2; append to cache
Token 3: retrieve cached K1, V1, K2, V2; compute only Q3, K3, V3; append to cache
...
Total: O(T) computation (linear!)
```

**Memory Layout:**
```
Cache for a transformer layer:
  key_cache[layer][batch_size][seq_len, d_k]      # shape: (B, T, d_k)
  value_cache[layer][batch_size][seq_len, d_v]    # shape: (B, T, d_v)

For each new token:
  new_k = compute(input_t)              # (B, 1, d_k)
  new_v = compute(input_t)              # (B, 1, d_v)
  key_cache[:, :, t:t+1, :] = new_k    # append new K
  value_cache[:, :, t:t+1, :] = new_v  # append new V
  
  # Attention uses full cached K, V (all previous + current)
  attn = softmax(Q @ cached_K^T) @ cached_V
```

**Impact on Latency:**
```
Without cache:
  Attention FLOPS = O(seq_len^2 × d)  per token
  
With cache:
  Attention FLOPS = O(seq_len × d)  per token
  
Speedup: O(seq_len) improvement! For seq_len=2048, ~2000x faster
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | No Cache | With Cache |
|--------|----------|-----------|
| FLOPS per token | O(T × d) | O(d) |
| Memory overhead | Minimal | O(T × L × d) |
| Latency per token | ~const | ~const ✓ |
| Throughput (batch) | High | Lower (cache memory limit) |
| Sequence length | Fast up to ~512 | Scales to 8k+ |
| Batch size | Limited by compute | Limited by memory (cache) |

**Sequence length tradeoff:**
- seq_len=100: cache ≈ 100MB
- seq_len=1000: cache ≈ 1GB
- seq_len=10000: cache ≈ 10GB (exceeds GPU memory for large models)

**Memory cost per token:**
```
For LLaMA 7B (4096 hidden_dim, 32 layers):
  per token, per layer = 4096 × 32 × 2 (for K and V) = 262KB per layer
  all 32 layers = 8.4MB per token
  
For seq_len=2048: 8.4MB × 2048 ≈ 17GB
```

### Code Implementation

```python
# Key imports for KV Cache
import numpy as np
import torch
from typing import Any

# KV Cache example implementation
class KVCache:
    """
    KV Cache implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is KV Cache used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of KV Cache?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does KV Cache compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using KV Cache?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind KV Cache?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/kv-cache.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]